In [ ]:
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.multitest as multitest

def load_json_file(file_path):
    """
    Load a JSON file and return its content as a Python object.

    :param file_path: Path to the JSON file.
    :return: Python object representing the JSON data.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

def save_json_file(data, file_path):
    """
    Save a Python object as a JSON file.

    :param data: Python object to be saved as JSON.
    :param file_path: Path where the JSON file will be saved.
    """
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

def print_json_structure(data):
    """
    Pretty print a Python object as JSON.

    :param data: Python object to be printed as JSON.
    """
    for dataset_key, dataset_value in data.items():
        print(f"Dataset: {dataset_key}")
        print(f"  Methods: {', '.join(dataset_value.keys())}")  # Print the remaining method keys
        print(f"  Metrics: {', '.join(dataset_value[next(iter(dataset_value))].keys())}")  # Print the metrics keys from the first method
        print()


# drop agentsimulator and unused metrics

In [11]:
# summary = load_json_file('results/summary.json')

# for dataset_key, dataset_value in summary.items():
#     print(f"Dataset: {dataset_key}")

#     del dataset_value['agentsimulator']  # Remove 'agentsimulator' key if it exists

#     print(f"  Methods: {', '.join(dataset_value.keys())}")  # Print the remaining method keys
#     print(f"  Metrics: {', '.join(dataset_value[next(iter(dataset_value))].keys())}")  # Print the metrics keys from the first method
#     print()
    
#     summary[dataset_key] = dataset_value  # Update the summary with the modified dataset_value

# save_json_file(summary, 'results/summary_modified.json')  # Save the modified summary back to the JSON file


# perform tests

In [12]:
# load prosit samples (ours vs paper) into the format used below: dataset -> method -> metric -> {mean, std, values}
dataset_convert = {
    'bpi2012': 'BPIC12W', 'BPI_Challenge_2012_W_Two_TS': 'BPIC12W',
    'bpi2017': 'BPIC17W', 'BPI_Challenge_2017_W_Two_TS': 'BPIC17W',
    'acr': 'ACR', 'ConsultaDataMining201618': 'ACR',
    'cvs': 'CVS', 'cvs_pharmacy': 'CVS',
    'p2p': 'P2P', 'confidential_2000': 'P2P',
}
method_convert = {'prosit-replication': 'prosit', 'prosit': 'prosit'}
metric_convert = {  # our metric names -> paper metric names
    'NGD': '2gd', 'CTD': 'ctd', 'CAR': 'car',
    'Rule-CFD': 'transition_tree', 'Rule-ATD': 'arrival_tree',
    'Rule-ETD': 'execution_tree', 'Rule-WTD': 'waiting_tree',
    'Execution Time Entropy': 'etd_entropy',
}

def rename(data, metric_map=None):
    """Keep only datasets/methods present in the convert dicts and rename dataset/method/metric keys."""
    out = {}
    for dataset_key, dataset_value in data.items():
        if dataset_key not in dataset_convert:
            continue
        for method_key, method_value in dataset_value.items():
            if method_key not in method_convert:  # e.g. simod, rims, dsim: not replicated
                continue
            out.setdefault(dataset_convert[dataset_key], {})[method_convert[method_key]] = {
                (metric_map or {}).get(metric_key, metric_key): metric_value
                for metric_key, metric_value in method_value.items()
            }
    return out

summary = rename(load_json_file('samples_ours_prosit.json'), metric_convert)
baseline = rename(load_json_file('samples_paper_prosit.json'))

print_json_structure(summary)
print_json_structure(baseline)


Dataset: P2P
  Methods: prosit
  Metrics: 2gd, ctd, car, transition_tree, arrival_tree, execution_tree, waiting_tree, etd_entropy

Dataset: CVS
  Methods: prosit
  Metrics: 2gd, ctd, car, transition_tree, arrival_tree, execution_tree, waiting_tree, etd_entropy

Dataset: ACR
  Methods: prosit
  Metrics: 2gd, ctd, car, transition_tree, arrival_tree, execution_tree, waiting_tree, etd_entropy

Dataset: BPIC12W
  Methods: prosit
  Metrics: 2gd, ctd, car, transition_tree, arrival_tree, execution_tree, waiting_tree, etd_entropy

Dataset: BPIC17W
  Methods: prosit
  Metrics: 2gd, ctd, car, transition_tree, arrival_tree, execution_tree, waiting_tree, etd_entropy

Dataset: BPIC12W
  Methods: prosit
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: BPIC17W
  Methods: prosit
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: P2P
  Methods: p

In [13]:
#t-tests

#           ngd    ctd    car    -----   entropy        Rule-ATD        Rule-CFD           Rule-ETD          -------          Rule-WTD
# metrics = ['2gd', 'ctd', 'car', '2rgd', 'etd_entropy', 'arrival_tree', 'transition_tree', 'execution_tree', 'resource_tree', 'waiting_tree']
metrics = ['2gd', 'ctd', 'car', 'etd_entropy', 'arrival_tree', 'transition_tree', 'execution_tree', 'waiting_tree']
datasets = ['BPIC12W', 'BPIC17W', 'ACR', 'CVS', 'P2P']
methods = ['prosit', 'simod', 'rims', 'dsim']

print(len(metrics)*len(datasets)*len(methods))


count_rejected = 0
count_tested = 0
p_values = []
for dataset_key, dataset_value in summary.items():
    for method_key, method_value in dataset_value.items():
        for metric_key, metric_value in method_value.items():
            if metric_key in metrics:
                print(f"Dataset: {dataset_key},     Method: {method_key},       Metric: {metric_key}")
                t_statistic = stats.ttest_ind_from_stats(
                    mean1=metric_value['mean'],
                    std1=metric_value['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                )
                count_tested += 1
                print(t_statistic)
                p_values.append(t_statistic.pvalue)
                if t_statistic.pvalue < 0.05:
                    print("Reject the null hypothesis: The means are significantly different.")
                    count_rejected += 1
                print()

print(f'Number of tests performed: {count_tested}')
print(f'Number of rejections: {count_rejected}')




160
Dataset: P2P,     Method: prosit,       Metric: 2gd
Ttest_indResult(statistic=69.075317881487, pvalue=3.510730828067365e-17)
Reject the null hypothesis: The means are significantly different.

Dataset: P2P,     Method: prosit,       Metric: ctd
Ttest_indResult(statistic=495.3328366841257, pvalue=3.6300889377866914e-29)
Reject the null hypothesis: The means are significantly different.

Dataset: P2P,     Method: prosit,       Metric: car
Ttest_indResult(statistic=32.56963796630242, pvalue=1.2994667015647943e-14)
Reject the null hypothesis: The means are significantly different.

Dataset: P2P,     Method: prosit,       Metric: transition_tree
Ttest_indResult(statistic=86.90172715025207, pvalue=1.3501326270521512e-14)
Reject the null hypothesis: The means are significantly different.

Dataset: P2P,     Method: prosit,       Metric: arrival_tree
Ttest_indResult(statistic=109.61302953524067, pvalue=1.2671384248650456e-15)
Reject the null hypothesis: The means are significantly different

In [14]:
combined_statistic, combined_p_value = stats.combine_pvalues(p_values, method='fisher')
print(f'Combined p-value (Fisher): {combined_p_value}')
if combined_p_value < 0.05:
    print("Reject the null hypothesis: The means are significantly different.")


holms_corrected = multitest.multipletests(p_values, alpha=0.05, method='holm')
print(holms_corrected[0])
num_rejections_holm = sum(holms_corrected[0])
print(f'Number of rejections after Holm-Bonferroni correction: {num_rejections_holm}')

Combined p-value (Fisher): 0.0
Reject the null hypothesis: The means are significantly different.
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True False  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]
Number of rejections after Holm-Bonferroni correction: 39


In [15]:
# table columns: Dataset, Method, Metric, Mean (Summary), Std (Summary), Mean (Baseline), Std (Baseline), t-statistic, p-value, holms p-value, Holm Reject Null Hypothesis
results_table = []
for dataset_key, dataset_value in summary.items():
    for method_key, method_value in dataset_value.items():
        for metric_key, metric_value in method_value.items():
            if metric_key in metrics:
                t_statistic = stats.ttest_ind_from_stats(
                    mean1=metric_value['mean'],
                    std1=metric_value['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                )
                holm_reject_null = holms_corrected[0][count_tested - len(p_values) + p_values.index(t_statistic.pvalue)]
                results_table.append({
                    'Dataset': dataset_key,
                    'Method': method_key,
                    'Metric': metric_key,
                    'Mean (Summary)': metric_value['mean'],
                    'Std (Summary)': metric_value['std'],
                    'Mean (Baseline)': baseline[dataset_key][method_key][metric_key]['mean'],
                    'Std (Baseline)': baseline[dataset_key][method_key][metric_key]['std'],
                    't-statistic': t_statistic.statistic,
                    'p-value': t_statistic.pvalue,
                    'Holm p-value': holms_corrected[1][count_tested - len(p_values) + p_values.index(t_statistic.pvalue)],
                    'Holm Reject Null Hypothesis': holm_reject_null
                })


In [16]:
print(results_table)

[{'Dataset': 'P2P', 'Method': 'prosit', 'Metric': '2gd', 'Mean (Summary)': 0.35959567855391045, 'Std (Summary)': 0.004570161814604642, 'Mean (Baseline)': 0.10575264941035153, 'Std (Baseline)': 0.010684593847586967, 't-statistic': 69.075317881487, 'p-value': 3.510730828067365e-17, 'Holm p-value': 9.384682139236405e-16, 'Holm Reject Null Hypothesis': True}, {'Dataset': 'P2P', 'Method': 'prosit', 'Metric': 'ctd', 'Mean (Summary)': 631.6976878612716, 'Std (Summary)': 3.569489408300847, 'Mean (Baseline)': 10.629012345679014, 'Std (Baseline)': 1.7262446940654328, 't-statistic': 495.3328366841257, 'p-value': 3.6300889377866914e-29, 'Holm p-value': 1.4520355751146766e-27, 'Holm Reject Null Hypothesis': True}, {'Dataset': 'P2P', 'Method': 'prosit', 'Metric': 'car', 'Mean (Summary)': 730.3710982658957, 'Std (Summary)': 28.06428647257354, 'Mean (Baseline)': 132.22808641975308, 'Std (Baseline)': 50.84433339215409, 't-statistic': 32.56963796630242, 'p-value': 1.2994667015647943e-14, 'Holm p-value':

# results table of means

In [17]:
print("""\\begin{table}[htbp]
\\centering
\\begin{tabular}{ll*{7}{>{\\centering\\arraybackslash}p{1.2cm}}>{\\centering\\arraybackslash}p{1.6cm}}
\\toprule
\\textbf{Method} & \\textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\\\""")
for dataset_key, dataset_value in summary.items():
    print("\\midrule")
    print(f"\\multicolumn{{9}}{{l}}{{{dataset_key}}}\\\\")
    for method_key, method_value in summary[dataset_key].items():
        lineprint_our = method_key + " & our"
        lineprint_baseline = method_key + " & paper"
        for metric_key in ['2gd', 'ctd', 'car', 'transition_tree', 'arrival_tree', 'execution_tree', 'waiting_tree', 'etd_entropy']:
            if metric_key in method_value and metric_key in baseline[dataset_key][method_key]:
                lineprint_our += f" & {method_value[metric_key]['mean']:.2f}"
                lineprint_baseline += f" & {baseline[dataset_key][method_key][metric_key]['mean']:.2f}"
                if holms_corrected[0][count_tested - len(p_values) + p_values.index(stats.ttest_ind_from_stats(
                    mean1=method_value[metric_key]['mean'],
                    std1=method_value[metric_key]['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                ).pvalue)]:
                    lineprint_our += " \\cellcolor{red!20}"
                    lineprint_baseline += " \\cellcolor{red!20}"
            else:
                lineprint_our += " & N/A"
                lineprint_baseline += " & N/A"
        print(lineprint_our + " \\\\")
        print(lineprint_baseline + " \\\\")
print("""\\bottomrule
\end{tabular}
\caption{Metric results of the author's simulation compared to its posted results. Cells highlighted in red indicate results are significantly different from the baseline (p < 0.05) after Holm-Bonferroni correction. N/A's are placed where no data was collected by the author, or the author's replication code was unable to produce results.}
\label{tab:results}
\end{table}""")

\begin{table}[htbp]
\centering
\begin{tabular}{ll*{7}{>{\centering\arraybackslash}p{1.2cm}}>{\centering\arraybackslash}p{1.6cm}}
\toprule
\textbf{Method} & \textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\
\midrule
\multicolumn{9}{l}{P2P}\\
prosit & our & 0.36 \cellcolor{red!20} & 631.70 \cellcolor{red!20} & 730.37 \cellcolor{red!20} & 0.32 \cellcolor{red!20} & 872.68 \cellcolor{red!20} & 24.22 \cellcolor{red!20} & 2469.39 \cellcolor{red!20} & 1.46 \cellcolor{red!20} \\
prosit & paper & 0.11 \cellcolor{red!20} & 10.63 \cellcolor{red!20} & 132.23 \cellcolor{red!20} & 0.00 \cellcolor{red!20} & 14.48 \cellcolor{red!20} & 4.03 \cellcolor{red!20} & 8.60 \cellcolor{red!20} & 1.87 \cellcolor{red!20} \\
\midrule
\multicolumn{9}{l}{CVS}\\
prosit & our & 0.02 \cellcolor{red!20} & 47.38 \cellcolor{red!20} & 550.17 \cellcolor{red!20} & 0.01 \cellcolor{red!20} & 34.58 \cellcolor{red!20} & 4.93 \cellcolor{red!20} & 77.17 \cellcolor{red!20} & 1.

In [18]:
# 
print("""\\begin{table}[htbp]
\\centering\\scriptsize
\\begin{tabular}{ll*{7}{>{\\centering\\arraybackslash}p{1.2cm}}>{\\centering\\arraybackslash}p{1.6cm}}
\\toprule
\\textbf{Method} & \\textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\\\""")
for dataset_key, dataset_value in summary.items():
    print("\\midrule")
    print(f"\\multicolumn{{9}}{{l}}{{{dataset_key}}}\\\\")
    for method_key, method_value in summary[dataset_key].items():
        lineprint_our = method_key + " & our"
        lineprint_baseline = method_key + " & paper"
        for metric_key in ['2gd', 'ctd', 'car', 'transition_tree', 'arrival_tree', 'execution_tree', 'waiting_tree', 'etd_entropy']:
            if metric_key in method_value and metric_key in baseline[dataset_key][method_key]:
                lineprint_our += f" & {method_value[metric_key]['mean']:.2f}"+"{\\tiny$\\pm$"+f"{method_value[metric_key]['std']:.2f}"+"}"
                lineprint_baseline += f" & {baseline[dataset_key][method_key][metric_key]['mean']:.2f}"+"{\\tiny$\\pm$"+f"{baseline[dataset_key][method_key][metric_key]['std']:.2f}"+"}"
                if holms_corrected[0][count_tested - len(p_values) + p_values.index(stats.ttest_ind_from_stats(
                    mean1=method_value[metric_key]['mean'],
                    std1=method_value[metric_key]['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                ).pvalue)]:
                    lineprint_our += " \\cellcolor{red!20}"
                    lineprint_baseline += " \\cellcolor{red!20}"
            else:
                lineprint_our += " & N/A"
                lineprint_baseline += " & N/A"
        print(lineprint_our + " \\\\")
        print(lineprint_baseline + " \\\\")
print("""\\bottomrule
\end{tabular}
\caption{Metric results of the author's simulation compared to its posted results. Cells highlighted in red indicate results are significantly different from the baseline (p < 0.05) after Holm-Bonferroni correction. N/A's are placed where no data was collected by the author, or the author's replication code was unable to produce results.}
\label{tab:results}
\end{table}""")

\begin{table}[htbp]
\centering\scriptsize
\begin{tabular}{ll*{7}{>{\centering\arraybackslash}p{1.2cm}}>{\centering\arraybackslash}p{1.6cm}}
\toprule
\textbf{Method} & \textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\
\midrule
\multicolumn{9}{l}{P2P}\\
prosit & our & 0.36{\tiny$\pm$0.00} \cellcolor{red!20} & 631.70{\tiny$\pm$3.57} \cellcolor{red!20} & 730.37{\tiny$\pm$28.06} \cellcolor{red!20} & 0.32{\tiny$\pm$0.01} \cellcolor{red!20} & 872.68{\tiny$\pm$24.64} \cellcolor{red!20} & 24.22{\tiny$\pm$2.77} \cellcolor{red!20} & 2469.39{\tiny$\pm$10.54} \cellcolor{red!20} & 1.46{\tiny$\pm$0.02} \cellcolor{red!20} \\
prosit & paper & 0.11{\tiny$\pm$0.01} \cellcolor{red!20} & 10.63{\tiny$\pm$1.73} \cellcolor{red!20} & 132.23{\tiny$\pm$50.84} \cellcolor{red!20} & 0.00{\tiny$\pm$0.00} \cellcolor{red!20} & 14.48{\tiny$\pm$2.45} \cellcolor{red!20} & 4.03{\tiny$\pm$0.32} \cellcolor{red!20} & 8.60{\tiny$\pm$1.39} \cellcolor{red!20} & 1.87{\tiny$